In [1]:
import os
os.environ["TRANSFORMERS_VERBOSITY"] = "info"

In [2]:
# imports
import json

import logging

# Set up logging configuration at the top of your notebook or script
logging.basicConfig(
    level=logging.DEBUG,  # Change to DEBUG for more verbosity
    format='%(asctime)s - %(levelname)s - %(message)s'
)

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-Coder-32B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
   model_name,
   dtype="auto",
   device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

2025-10-12 12:00:11,992 - DEBUG - Starting new HTTPS connection (1): huggingface.co:443
2025-10-12 12:00:12,125 - DEBUG - https://huggingface.co:443 "HEAD /Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/config.json HTTP/1.1" 307 0
2025-10-12 12:00:12,131 - DEBUG - https://huggingface.co:443 "HEAD /api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/config.json HTTP/1.1" 200 0
loading configuration file config.json from cache at /workspace/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-32B-Instruct/snapshots/381fc969f78efac66bc87ff7ddeadb7e73c218a7/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 5120,
  "initializer_range": 0.02,
  "intermediate_size": 27648,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attent

Loading checkpoint shards:   0%|          | 0/14 [00:00<?, ?it/s]

2025-10-12 12:00:22,072 - DEBUG - https://huggingface.co:443 "HEAD /Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/generation_config.json HTTP/1.1" 307 0
2025-10-12 12:00:22,078 - DEBUG - https://huggingface.co:443 "HEAD /api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/generation_config.json HTTP/1.1" 200 0
loading configuration file generation_config.json from cache at /workspace/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-32B-Instruct/snapshots/381fc969f78efac66bc87ff7ddeadb7e73c218a7/generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "repetition_penalty": 1.05,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}

2025-10-12 12:00:22,186 - DEBUG - https://huggingface.co:443 "HEAD /Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/custom_generate/generate.py HTTP/1.1" 404 0
Could not locate the custom_gene

In [4]:
import time

def generate_for_prompt(model, tokenizer, user_prompt, **kwargs):
    def build_messages(user_prompt):
        return [
            {
                "role": "system",
                "content": (
                    "Output only the Java code, with no explanations or comments."
                )
            },
            {"role": "user", "content": user_prompt}
        ]

    logging.info(kwargs)
    max_new_tokens = kwargs.get("max_new_tokens", 1024)
    top_p = kwargs.get("top_p", 0.95)
    temperature = kwargs.get("temperature", 0.1)
    top_k = kwargs.get("top_k", 0)

    num_return_sequences = kwargs.get("num_return_sequences", 1)
    do_sample = kwargs.get("do_sample", False)

    messages = build_messages(user_prompt)
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    # Measure execution time
    start_time = time.time()
    
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        num_return_sequences=num_return_sequences,
        top_p=top_p,
        temperature=temperature,
        top_k=top_k
    )

    execution_time = time.time() - start_time
    
    logging.info(f"⏱️ Took {execution_time:.2f} secs")

    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(
        generated_ids, skip_special_tokens=True)[0]
    return response

In [5]:
from tqdm import tqdm
import math
from concurrent.futures import ThreadPoolExecutor, as_completed


def generate_for_prompts_v2(model, tokenizer, prompts, chunk_size=4, max_workers=4, **kwargs):
    """
    Generate completions for prompts in parallel, divided into chunks.
    Reports overall progress across all prompts using tqdm.
    """
    total = len(prompts)
    completions = [None] * total
    num_chunks = math.ceil(total / chunk_size)
    logging.info(
        f"Total prompts: {total}, Chunk size: {chunk_size}, Chunks: {num_chunks}")

    def process_chunk(chunk_prompts, chunk_indices):
        chunk_results = []
        for idx, prompt in zip(chunk_indices, chunk_prompts):
            response = generate_for_prompt(model, tokenizer, prompt, **kwargs)
            chunk_results.append((idx, response))
        return chunk_results

    # Prepare chunks
    chunks = [
        (prompts[i:i+chunk_size], list(range(i, min(i+chunk_size, total))))
        for i in range(0, total, chunk_size)
    ]

    with ThreadPoolExecutor(max_workers=max_workers) as executor, tqdm(total=total, desc="Overall Progress") as pbar:
        futures = {executor.submit(process_chunk, chunk_prompts, chunk_indices): (
            chunk_prompts, chunk_indices) for chunk_prompts, chunk_indices in chunks}
        for future in as_completed(futures):
            chunk_results = future.result()
            for idx, response in chunk_results:
                completions[idx] = response
                pbar.update(1)

    logging.info("All completions finished.")
    return completions

In [6]:
def generate_for_prompts(model, tokenizer, prompts, **kwargs):
    completions = []
    start_time = time.time()  # Record start time
    for idx, prompt in enumerate(prompts):
        logging.info(f"Generating completion for problem {idx+1}/{len(prompts)}")
        response = generate_for_prompt(model, tokenizer, prompt, **kwargs)
        completions.append(response if isinstance(
            response, list) else [response])

        logging.info(f'Completion for problem {idx+1}/{len(prompts)}\n')
        logging.debug(f'Completion for problem {idx+1}/{len(prompts)}:\n{response}\n')
        
        # total time has passed
        elapsed_time = time.time() - start_time
        logging.info(f'Elapsed time: {elapsed_time:.2f} secs')
        logging.info(f'Estimated time remaining: {(elapsed_time/(idx+1))*(len(prompts)-idx+1):.2f} secs')
        logging.info(f'{"-"*40}\n')

    return completions

In [7]:
def generate_for_dataset(model, tokenizer, problems, **kwargs):
    user_prompts = [problem['prompt'] for problem in problems]
    
    completions = generate_for_prompts_v2(model, tokenizer, user_prompts, **kwargs) if kwargs.get(
        'parallel', False) else generate_for_prompts(model, tokenizer, user_prompts, **kwargs)
    
    # Save completions to a JSON file
    max_new_tokens = kwargs.get("max_new_tokens", 1024)
    top_p = kwargs.get("top_p", 0.95)
    temperature = kwargs.get("temperature", 0.1)
    top_k = kwargs.get("top_k", 0)
    num_return_sequences = kwargs.get("num_return_sequences", 1)
    do_sample = kwargs.get("do_sample", False)
    output_path = f'mnt{max_new_tokens}_p{top_p}_t{temperature}_k{top_k}_seq{num_return_sequences}_sampling{do_sample}_completions.json'
    
    with open(output_path, 'w') as f:
        json.dump(completions, f, indent=4)

    return completions

In [8]:
test_prompt = """
import java.util.*;
import java.lang.*;

class Solution {
    /**
        Given a positive floating point number, it can be decomposed into
        and integer part (largest integer smaller than given number) and decimals
        (leftover part always smaller than 1).

        Return the decimal part of the number.
        >>> truncateNumber(3.5)
        0.5
     */
    public double truncateNumber(double number) {        
""";

In [9]:
logging.info(generate_for_prompt(model, tokenizer, test_prompt))

2025-10-12 12:00:22,637 - INFO - {}
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k'].
- `temperature`: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
- `top_p`: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
- `top_k`: `do_sample` is set to `False`. However, `top_k` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
If you're using a pretrained model, note that some of these attributes may be set through the model's `generation_config.json` file.
2025-10-12 12:00:23,716 - INFO - ⏱️ Took 1.07 secs
2025-10-12 12:00:23,716 - INFO - ```java
        return number - Math.floor(number);
    }
}
```


In [10]:
import os
print(os.getcwd())
ds_json_path = os.path.join(
    '../../../../../..', 'benchmark/datasets/humaneval-x/humanevalx-java-refined.json')

problems = json.load(open(ds_json_path, 'r'))
logging.info(f'Loaded {len(problems)} problems from "{ds_json_path}"')

# 0.2 0.95 topk=0
generate_for_dataset(model, tokenizer, problems, parallel=False, chunk_size=2, max_workers=4,
                    do_sample=True, max_new_tokens=1024, top_p=0.95, temperature=0.2, top_k=0, seed=10)

2025-10-12 12:00:23,721 - INFO - Loaded 164 problems from "../../../../../../benchmark/datasets/humaneval-x/humanevalx-java-refined.json"
2025-10-12 12:00:23,722 - INFO - Generating completion for problem 1/164
2025-10-12 12:00:23,722 - INFO - {'parallel': False, 'chunk_size': 2, 'max_workers': 4, 'do_sample': True, 'max_new_tokens': 1024, 'top_p': 0.95, 'temperature': 0.2, 'top_k': 0, 'seed': 10}


/workspace/bigcode-evaluation-harness/benchmark/Qwen2.5-Coder-32B-Instruct/java/improve/pass@1/t0.2-p0.95-k0-batch1-n1


2025-10-12 12:00:26,839 - INFO - ⏱️ Took 3.12 secs
2025-10-12 12:00:26,840 - INFO - Completion for problem 1/164

2025-10-12 12:00:26,840 - DEBUG - Completion for problem 1/164:
        Collections.sort(numbers);
        for (int i = 0; i < numbers.size() - 1; i++) {
            if (Math.abs(numbers.get(i) - numbers.get(i + 1)) < threshold) {
                return true;
            }
        }
        return false;
    }
}

2025-10-12 12:00:26,840 - INFO - Elapsed time: 3.12 secs
2025-10-12 12:00:26,840 - INFO - Estimated time remaining: 514.55 secs
2025-10-12 12:00:26,840 - INFO - ----------------------------------------

2025-10-12 12:00:26,841 - INFO - Generating completion for problem 2/164
2025-10-12 12:00:26,841 - INFO - {'parallel': False, 'chunk_size': 2, 'max_workers': 4, 'do_sample': True, 'max_new_tokens': 1024, 'top_p': 0.95, 'temperature': 0.2, 'top_k': 0, 'seed': 10}
2025-10-12 12:00:33,090 - INFO - ⏱️ Took 6.25 secs
2025-10-12 12:00:33,091 - INFO - Completion for proble

[['        Collections.sort(numbers);\n        for (int i = 0; i < numbers.size() - 1; i++) {\n            if (Math.abs(numbers.get(i) - numbers.get(i + 1)) < threshold) {\n                return true;\n            }\n        }\n        return false;\n    }\n}'],
 ["```java\n        List<String> result = new ArrayList<>();\n        int balance = 0;\n        StringBuilder currentGroup = new StringBuilder();\n\n        for (char c : paren_string.toCharArray()) {\n            if (c == ' ') {\n                continue;\n            }\n            currentGroup.append(c);\n            if (c == '(') {\n                balance++;\n            } else if (c == ')') {\n                balance--;\n            }\n            if (balance == 0 && currentGroup.length() > 0) {\n                result.add(currentGroup.toString());\n                currentGroup.setLength(0);\n            }\n        }\n\n        return result;\n    }\n}\n```"],
 ['        return number - Math.floor(number);\n    }\n}'],
 

In [11]:
import json

# Evaluation
generations_path = 'mnt1024_p0.95_t0.2_k0_seq1_samplingTrue_completions.json'

preprocessed_generations_path = generations_path.replace(".json", '_preprocessed.json')

orig_gens = json.load(open(generations_path, 'r'))
processed_gens = []

for gens in orig_gens:
    new_gens = []
    for gen in gens:
        # replace the last occurrence of `\n    }\n}`
        gen = gen.rsplit('\n    }\n}', 1)[0]
        new_gens.append(gen)
    
    processed_gens.append(new_gens)
    
with open(preprocessed_generations_path, 'w') as f:
    json.dump(processed_gens, f, indent=4)
    
logging.info(f'Saved preprocessed generations to "{preprocessed_generations_path}"')

2025-10-12 12:12:53,436 - INFO - Saved preprocessed generations to "mnt1024_p0.95_t0.2_k0_seq1_samplingTrue_completions_preprocessed.json"
